In [5]:
import os
import img2pdf

IMAGES_FOLDER = "../data/pub/images"
# This will automatically create a subfolder for the new PDFs
PDF_FOLDER = "../data/pub/pdf_converted"

if not os.path.exists(PDF_FOLDER):
    os.makedirs(PDF_FOLDER)

# List of images in the folder
images = [f for f in os.listdir(IMAGES_FOLDER) if f.lower().endswith((".jpg", ".jpeg"))]

# Processing images
if not images:
    print("No .jpg images found in the specified folder.")
else:
    print(f"Converting {len(images)} images to PDF\n")
    for file in images:
        image_path = os.path.join(IMAGES_FOLDER, file)
        # Extract the file name without the .jpg extension
        name_without_ext, _ = os.path.splitext(file)
        output_pdf_path = os.path.join(PDF_FOLDER, f"{name_without_ext}.pdf")
        try:
            # Open a new PDF file in binary write mode ("wb")
            with open(output_pdf_path, "wb") as pdf_file:
                # img2pdf injects the raw JPG data directly into the PDF container
                pdf_file.write(img2pdf.convert(image_path))
            print(f"PDF created: {name_without_ext}.pdf")
        except Exception as e:
            print(f"Error converting {file}: {e}")

    print("-" * 30)
    print(f"Process finished successfully. Check the '{PDF_FOLDER}' folder.")

Converting 42 images to PDF

PDF created: PMC1064098_table_2.pdf
PDF created: PMC1064076_table_1.pdf
PDF created: PMC1064101_table_1.pdf
PDF created: PMC1064094_table_1.pdf
PDF created: PMC1064095_table_0.pdf
PDF created: PMC1064098_table_0.pdf
PDF created: PMC1064076_table_2.pdf
PDF created: PMC1064094_table_0.pdf
PDF created: PMC1064078_table_2.pdf
PDF created: PMC1064098_table_3.pdf
PDF created: PMC1064074_table_0.pdf
PDF created: PMC1064078_table_4.pdf
PDF created: PMC1064094_table_2.pdf
PDF created: PMC1064081_table_1.pdf
PDF created: PMC1064081_table_0.pdf
PDF created: PMC1064100_table_1.pdf
PDF created: PMC1064076_table_0.pdf
PDF created: PMC1064082_table_0.pdf
PDF created: PMC1064100_table_3.pdf
PDF created: PMC1064078_table_3.pdf
PDF created: PMC1064097_table_2.pdf
PDF created: PMC1064095_table_2.pdf
PDF created: PMC1064097_table_1.pdf
PDF created: PMC1064097_table_3.pdf
PDF created: PMC1064095_table_3.pdf
PDF created: PMC1064082_table_1.pdf
PDF created: PMC1064078_table_0.pdf

In [ ]:
from src.data_treatment import normalize_pub

d = {"extractions": [{"description": "ROANOKE\u2664WROV\n-FM"}]}


json_1 = {
    "extractions": [{"line_item_description": "ROANOKE\nWROV-FM", "date": "20-05-2020"}]
}


clean = normalize_pub(json_1)
print(clean)

{'extractions': [{'line_item_description': 'ROANOKE WROV-FM', 'date': '20-05-2020'}]}


In [3]:
import json


# --- 1. La lógica pura de limpieza de texto ---
def limpiar_texto(texto):
    """Aplica las reglas de limpieza a un texto individual."""
    if not isinstance(texto, str):
        return texto
    # Quitar las dobles barras si las hubiera
    limpio = texto.replace("\\n", " ")
    # Planchar cualquier salto de línea (\n, \r) y espacios múltiples
    limpio = " ".join(limpio.split())
    # Sustituir guiones y símbolos especiales
    limpio = (
        limpio.replace("\u2013", "-").replace("\u2664", "<=").replace("\u2665", ">=")
    )
    return limpio


# --- 2. El explorador ciego (Recursividad) ---
def recorrer_y_limpiar(datos):
    """Viaja por todo el JSON buscando strings para limpiarlos, sin importar las claves."""
    # CASO A: Es un diccionario -> miramos todas sus claves y valores
    if isinstance(datos, dict):
        for clave, valor in datos.items():
            # Volvemos a llamar a la función para cada valor
            datos[clave] = recorrer_y_limpiar(valor)
    # CASO B: Es una lista -> miramos todos sus elementos
    elif isinstance(datos, list):
        for i in range(len(datos)):
            # Volvemos a llamar a la función para cada elemento
            datos[i] = recorrer_y_limpiar(datos[i])
    # CASO C: Hemos llegado al fondo de la caja y es un texto -> ¡Lo limpiamos!
    elif isinstance(datos, str):
        return limpiar_texto(datos)
    # CASO D: Es un número, booleano, nulo, etc -> Lo devolvemos sin tocar
    return datos


# --- 3. La función principal (La puerta de entrada) ---
def normalize_general(data):
    """Asegura que los datos estén bien formateados antes de soltar al explorador."""
    if isinstance(data, str):
        try:
            data = json.loads(data)
        except json.JSONDecodeError:
            pass  # Si es un string normal (no un JSON), seguimos adelante

    return recorrer_y_limpiar(data)


datos_procesados = normalize_general(json_1)

print(json.dumps(datos_procesados, indent=4))

{
    "extractions": [
        {
            "line_item_description": "ROANOKE WROV-FM",
            "date": "20-05-2020"
        }
    ]
}
